In [45]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import ruamel.yaml
import math
import numpy as np
import tarfile
from matplotlib import cm
from matplotlib import pyplot as plt
import warnings
import yaml
yaml_format = ruamel.yaml.YAML()
warnings.filterwarnings('ignore')
from matplotlib.ticker import FuncFormatter  # Add this import at the top
import pickle  # Add this import at the top
import matplotlib.ticker as mticker


import matplotlib.ticker as ticker  # Add this import at the top
OUTPUT_DIR = './figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

app_name_mapping = {
    'ones-npb-ft': 'NPB-FT',
    'ones-npb-is': 'NPB-IS',
    'ones-stream-triad': 'Stream-Triad',
    'ones-stream-scale': 'Stream-Scale',
    'ones-stream-copy': 'Stream-Copy',
    'ones-stream-add': 'Stream-Add',
    'ones-stream-full': 'Stream-Full',
    'ones-npb-ep': 'NPB-EP',
    'phases-stream-full': 'Stream-Phase',
    'ones-npb-mg': 'NPB-MG',
    'ones-npb-cg': 'NPB-CG',
    'ones-npb-bt': 'NPB-BT'
}

In [46]:
def get_data_dir(subfolder):
    # current_dir = os.path.dirname(os.path.abspath(__file__))
    current_dir = os.getcwd()
    print(current_dir)
    return os.path.join(current_dir, "experiment_data" ,subfolder)


In [47]:
def derived_papi(PAPI_data):
    DERIVED = {}
    min_length = min(len(PAPI_data['PAPI_TOT_INS']['instantaneous_value']), len(PAPI_data['PAPI_TOT_CYC']['instantaneous_value']))
    DERIVED['TOT_INS_PER_CYC'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_TOT_INS']['instantaneous_value'][:min_length]) / np.array(PAPI_data['PAPI_TOT_CYC']['instantaneous_value'].iloc[:min_length]),
        'timestamp': np.array(PAPI_data['PAPI_TOT_INS'].time)
    })
    DERIVED['TOT_CYC_PER_INS'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_TOT_CYC']['instantaneous_value']) / np.array(PAPI_data['PAPI_TOT_INS']['instantaneous_value']),
        'timestamp': np.array(PAPI_data['PAPI_TOT_CYC'].time)
    })
    DERIVED['L3_TCM_PER_TCA'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_L3_TCM']['instantaneous_value']) / np.array(PAPI_data['PAPI_L3_TCA']['instantaneous_value']),
        'timestamp': np.array(PAPI_data['PAPI_L3_TCM'].time)
    })  
    DERIVED['TOT_STL_PER_CYC'] = pd.DataFrame({
        'value': np.array(PAPI_data['PAPI_RES_STL']['instantaneous_value']) / np.array(PAPI_data['PAPI_TOT_CYC']['instantaneous_value']),
        'timestamp': np.array(PAPI_data['PAPI_RES_STL'].time)
    })
    
    # Create a mask for non-NaN values across all keys
    mask = ~DERIVED['TOT_INS_PER_CYC']['value'].isna()  # Start with one key to create the mask
    for key in DERIVED:
        mask &= ~DERIVED[key]['value'].isna()  # Combine masks for all keys

    for key in DERIVED:
        DERIVED[key] = DERIVED[key][mask]  # Filter each DataFrame using the combined mask

    return DERIVED

def progress_per_cycle(PAPI_data, progress_data):
    PROGRESS_PER_CYCLE = {}
    min_length = min(len(PAPI_data['PAPI_TOT_CYC']['instantaneous_value']), len(progress_data['value']))
    PROGRESS_PER_CYCLE['progress_per_cycle'] = pd.DataFrame({
        'value': np.array(progress_data['value'][:min_length]) / np.array(PAPI_data['PAPI_TOT_CYC']['instantaneous_value'][:min_length]),
        'timestamp': np.array(progress_data['time'][:min_length])
    })
    return PROGRESS_PER_CYCLE

def collect_power(power_data):
    power={}
    power['geopm_0'] = power_data[power_data.scope==0]
    power['geopm_1'] = power_data[power_data.scope==1]
    power['geopm_0']['elapsed_time'] = power['geopm_0']['time'] - power['geopm_0']['time'].iloc[0]
    power['geopm_1']['elapsed_time'] = power['geopm_1']['time'] - power['geopm_1']['time'].iloc[0]
    
    min_length = min(len(power['geopm_0']), len(power['geopm_1']))
    geopm_power_0 = power['geopm_0'][:min_length]
    geopm_power_1 = power['geopm_1'][:min_length]

    average_power = pd.DataFrame({
        'time_stamp': geopm_power_0['time'],  # Use the timestamp from geopm_power_0
        'average_power': [(p0 + p1) / 2 for p0, p1 in zip(geopm_power_0['value'], geopm_power_1['value'])]
    })
    average_power['elapsed_time'] = power['geopm_0']['elapsed_time']
    power['average_power'] = average_power
    return power

def calculate_power_with_wraparound(current, previous, time_diff, wraparound_value=262143.328850):
    diff = current - previous
    if diff < 0:  # Wraparound detected
        diff = (wraparound_value - previous) + current
    return diff / time_diff

def compute_power(pubEnergy,power_data=None):
    power = {}
    geopm_sensor0 = geopm_sensor1 = pd.DataFrame({'timestamp':[],'value':[]})
    for i,row in pubEnergy.iterrows():
        if i%2 == 0:
            geopm_sensor0 = pd.concat([geopm_sensor0, pd.DataFrame([{'timestamp': row['time'], 'value': row['value']}])], ignore_index=True)
        else:
            geopm_sensor1 = pd.concat([geopm_sensor1, pd.DataFrame([{'timestamp': row['time'], 'value': row['value']}])], ignore_index=True)


    power['geopm_power_0'] = pd.DataFrame({
        'timestamp': geopm_sensor0['timestamp'][1:],  # Add timestamps
        'power': [
            calculate_power_with_wraparound(
                geopm_sensor0['value'][i],
                geopm_sensor0['value'][i-1],
                geopm_sensor0['timestamp'][i] - geopm_sensor0['timestamp'][i-1]
            ) for i in range(1, len(geopm_sensor0))
        ]
    })

    # Apply the same logic to geopm_power_1
    power['geopm_power_1'] = pd.DataFrame({
        'timestamp': geopm_sensor1['timestamp'][1:],  # Add timestamps
        'power': [
            calculate_power_with_wraparound(
                geopm_sensor1['value'][i],
                geopm_sensor1['value'][i-1],
                geopm_sensor1['timestamp'][i] - geopm_sensor1['timestamp'][i-1]
            ) for i in range(1, len(geopm_sensor1))
        ]
    })

    min_length = min(len(power['geopm_power_0']), len(power['geopm_power_1']))
    geopm_power_0 = power['geopm_power_0'][:min_length]
    geopm_power_1 = power['geopm_power_1'][:min_length]
    # fig,axs = plt.subplots(3,1)
    # axs[0].plot(geopm_power_0['timestamp'], geopm_power_0['power'], label='Node 0')
    # axs[1].plot(geopm_power_1['timestamp'], geopm_power_1['power'], label='Node 1')

    average_power = pd.DataFrame({
        'timestamp': geopm_power_0['timestamp'],  # Use the timestamp from geopm_power_0
        'average_power': [(p0 + p1) / 2 for p0, p1 in zip(geopm_power_0['power'], geopm_power_1['power'])]
    })
    average_power['elapsed_time'] = average_power['timestamp'] - average_power['timestamp'].iloc[0]
    # axs[2].plot(average_power['timestamp'], average_power['average_power'], label='Average Power', color='green')
    power['average_power'] = average_power
    return power

def measure_progress(progress_data, energy_data):
    Progress_DATA = {} 
    progress_sensor = pd.DataFrame(progress_data)
    first_sensor_point = min(energy_data['average_power']['timestamp'].iloc[0], progress_sensor['time'][0])
    progress_sensor['elapsed_time'] = progress_sensor['time'] - first_sensor_point  # New column for elapsed time
    # progress_sensor = progress_sensor.set_index('elapsed_time')
    performance_elapsed_time = progress_sensor.elapsed_time
    # Add performance_frequency as a new column in progress_sensor
    frequency_values = [
        progress_data['value'].iloc[t] / (performance_elapsed_time[t] - performance_elapsed_time[t-1]) for t in range(1, len(performance_elapsed_time))
    ]
    
    # Ensure the frequency_values length matches the index length
    frequency_values = [0] + frequency_values  # Prepend a 0 for the first index
    progress_sensor['frequency'] = frequency_values
    upsampled_timestamps= energy_data['average_power']['timestamp']
    
    # true_count = (progress_sensor['time'] <= upsampled_timestamps.iloc[0]).sum()

    progress_frequency_median = pd.DataFrame({'median': np.nanmedian(progress_sensor['frequency'].where(progress_sensor['time'] <= upsampled_timestamps.iloc[0])), 'timestamp': upsampled_timestamps.iloc[0]}, index=[0])
    for t in range(1, len(upsampled_timestamps)):
        progress_frequency_median = pd.concat([progress_frequency_median, pd.DataFrame({'median': [np.nanmedian(progress_sensor['frequency'].where((progress_sensor['time'] >= upsampled_timestamps.iloc[t-1]) & (progress_sensor['time'] <= upsampled_timestamps.iloc[t])))],
        'timestamp': [upsampled_timestamps.iloc[t]]})], ignore_index=True)
    progress_frequency_median['elapsed_time'] = progress_frequency_median['timestamp'] - progress_frequency_median['timestamp'].iloc[0]
    # Assign progress_frequency_median as a new column
    Progress_DATA['progress_sensor'] = progress_sensor
    Progress_DATA['progress_frequency_median'] = progress_frequency_median
    return Progress_DATA

def collect_papi(PAPI_data):
    PAPI = {}
    for scope in PAPI_data['scope'].unique():
        # Extract the string between the 3rd and 4th dots
        scope_parts = scope.split('.')
        if len(scope_parts) > 4:  # Ensure there are enough parts
            extracted_scope = scope_parts[3]
            # Aggregate the data for the extracted scope using pd.concat
            PAPI[extracted_scope] = PAPI_data[PAPI_data['scope'] == scope]
            instantaneous_values = [0] + [PAPI[extracted_scope]['value'].iloc[k] - PAPI[extracted_scope]['value'].iloc[k-1] for k in range(1,len(PAPI[extracted_scope]))]
            # Normalize the instantaneous values between 0 and 10
            # min_val = min(instantaneous_values)
            # max_val = max(instantaneous_values)
            PAPI[extracted_scope]['instantaneous_value'] = instantaneous_values
            PAPI[extracted_scope]['elapsed_time'] = PAPI[extracted_scope]['time'] - PAPI[extracted_scope]['time'].iloc[0]
    return PAPI


def generate_PCAP(PCAP_data):
    for row in PCAP_data.iterrows():
        if row[1]['time'] == 0:
            PCAP_data = PCAP_data.drop(row[0])


    PCAP_data['elapsed_time'] = PCAP_data['time'] - PCAP_data['time'].iloc[0]
    return PCAP_data


def compute_energy_consump(test_data_frame):
    total_energy = 0
    for i,t in enumerate(test_data_frame['average_power']['elapsed_time'].iloc[:-1]):
        total_energy += test_data_frame['average_power']['average_power'].iloc[i] * (test_data_frame['average_power']['elapsed_time'].iloc[i+1]-test_data_frame['average_power']['elapsed_time'].iloc[i])
    return total_energy


In [48]:
DATA_DIR = get_data_dir('plotting_data')
root,folders,files = next(os.walk(DATA_DIR))
test_results = {}
for APP in folders:
    print(APP)
    APP_DIR = os.path.join(DATA_DIR, APP)
    test_results[APP] = {}
    for tar_file in next(os.walk(APP_DIR))[2]:
        test_results[APP][tar_file] = {}
        if tar_file.endswith('.tar'):
            tar_path = os.path.join(APP_DIR, tar_file)
            extract_dir = os.path.join(APP_DIR, tar_file[:-4])  
            
            if not os.path.exists(extract_dir):
                os.makedirs(extract_dir)
            
            with tarfile.open(tar_path, 'r') as tar:
                tar.extractall(path=extract_dir)
        
        pubProgress = pd.read_csv(f'{extract_dir}/progress.csv')
        pubEnergy = pd.read_csv(f'{extract_dir}/energy.csv')
        pubPAPI = pd.read_csv(f'{extract_dir}/papi.csv')
        pubPCAP = pd.read_csv(f'{extract_dir}/PCAP_file.csv')
        pubPower = pd.read_csv(f'{extract_dir}/measured_power.csv')

        test_results[APP][tar_file]['energy'] = pubEnergy
        test_results[APP][tar_file]['power'] = compute_power(pubEnergy)
        test_results[APP][tar_file]['progress'] = measure_progress(pubProgress,test_results[APP][tar_file]['power'])
        test_results[APP][tar_file]['papi'] = collect_papi(pubPAPI)
        test_results[APP][tar_file]['PCAP'] = generate_PCAP(pubPCAP)
        test_results[APP][tar_file]['derived_papi'] = derived_papi(test_results[APP][tar_file]['papi'])   
        test_results[APP][tar_file]['pubPower'] = pubPower
        test_results[APP][tar_file]['progress_per_cycle'] = progress_per_cycle(test_results[APP][tar_file]['papi'], test_results[APP][tar_file]['progress']['progress_sensor'])
# print(test_results[APP][tar_file])

/Users/akhileshraj/Documents/summer2024/main_codes
ones-stream-add
ones-npb-ft
ones-stream-scale
ones-npb-bt
ones-stream-triad
ones-stream-full
phases-stream-full
ones-npb-ep
ones-npb-cg
ones-npb-is
ones-stream-copy
ones-npb-mg


In [49]:
for app in test_results.keys():
    for trace in test_results[app].keys():
        print(test_results[app][trace]['PCAP'])

           time                           actuator  value  elapsed_time
0  1.765031e+09  Actuator(ptr=c_void_p(234370272))  101.0      0.000000
1  1.765031e+09  Actuator(ptr=c_void_p(234370272))  101.0     10.042573
2  1.765031e+09  Actuator(ptr=c_void_p(234370272))  101.0     20.089451
3  1.765031e+09  Actuator(ptr=c_void_p(234370272))  101.0     30.142904
4  1.765031e+09  Actuator(ptr=c_void_p(234370272))  101.0     40.190300
5  1.765031e+09  Actuator(ptr=c_void_p(234370272))  101.0     50.239168
           time                           actuator  value  elapsed_time
0  1.765109e+09  Actuator(ptr=c_void_p(234370272))  141.0      0.000000
1  1.765109e+09  Actuator(ptr=c_void_p(234370272))  141.0     10.044140
2  1.765109e+09  Actuator(ptr=c_void_p(234370272))  141.0     20.095244
3  1.765109e+09  Actuator(ptr=c_void_p(234370272))  141.0     30.141662
4  1.765109e+09  Actuator(ptr=c_void_p(234370272))  141.0     40.184623
5  1.765109e+09  Actuator(ptr=c_void_p(234370272))  141.0     50

In [50]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial import distance
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
from time import time
import seaborn as sns
import matplotlib.pyplot as plt
import math

def simi_cosine(X, f):
    ''' 
    X is the data matrix, rows are applications, columns are events
    f is the path for X, used for graph title 
    '''  
    Appname = [app_name_mapping.get(name, name) for name in X.index]
    X = X.loc[:, (X != 0).any(axis=0)]  # drop the zero columns does not change much
    t1 = time()
    cosX = cosine_similarity(X)
    t2 = time()
    print("cost time:", t2 - t1)
    cosX = np.round(cosX, decimals=6)

    degree = [list(map(lambda x: math.degrees(math.acos(x)), line)) for line in cosX]  # show the degree of cosine

    plt.figure(figsize=(24, 18))
    sns.set(font_scale=2.5)
    sns.set_style("ticks")

    # lower triangular heatmap with diagonal
    mask = np.ones_like(degree)
    mask[np.tril_indices_from(mask)] = False

    ax = sns.heatmap(degree, mask=mask, annot=True, annot_kws={'size': 20}, fmt=".0f", cbar=True, linewidths=.5, cmap="RdYlGn_r", vmax=90, vmin=0)
    ax.set_yticklabels(Appname, rotation=0) 
    ax.set_xticklabels(Appname, ha="right", rotation=40) 
    ax.set_title("cosine similarity " + f)
    # ax.invert_yaxis()
    plt.savefig(f"{OUTPUT_DIR}/cosine_similarity_{f}.png")
    plt.close()

def simi_JS(X, f):
    ''' 
    X is the data matrix, rows are applications, columns are events
    f is the path for X, used for graph title 
    ''' 
    Appname = [app_name_mapping.get(name, name) for name in X.index]
    X_value = X.values
    jsd = [[0] * len(X) for _ in range(len(X_value))]
    t1 = time()
    for i in range(len(X_value)):
        for j in range(len(X_value)):
            jsd[i][j] = distance.jensenshannon(X_value[i], X_value[j]) * 100
    t2 = time()
    print("cost time:", t2 - t1)

    plt.figure(figsize=(24, 18))
    sns.set(font_scale=2.5)
    sns.set_style("ticks")

    mask = np.ones_like(jsd)
    mask[np.tril_indices_from(mask)] = False
    ax = sns.heatmap(jsd, mask=mask, annot=True, fmt='.0f', annot_kws={'size': 20}, cbar=True, linewidths=.5, cmap="RdYlGn_r", vmax=90, vmin=0)
    ax.set_yticklabels(Appname, rotation=0) 
    ax.set_xticklabels(Appname, ha="right", rotation=40) 
    ax.set_title("JS-divergence " + f)
    # ax.invert_yaxis()
    plt.savefig(f"{OUTPUT_DIR}/JS_divergence_{f}.png")
    plt.close()

def simi_wd(X, f):
    ''' 
    X is the data matrix, rows are applications, columns are events
    f is the path for X, used for graph title 
    ''' 
    Appname = [app_name_mapping.get(name, name) for name in X.index]
    X_value = X.values
    wd = [[0] * len(X) for _ in range(len(X_value))]
    t1 = time()
    for i in range(len(X_value)):
        for j in range(len(X_value)):
            wd[i][j] = wasserstein_distance(X_value[i], X_value[j]) * 1000
            # wd[i][j] = round(wd[i][j], -1)  # Round to nearest 10
    t2 = time()
    print("cost time:", t2 - t1)

    plt.figure(figsize=(24, 18))
    sns.set(font_scale=2.5)
    sns.set_style("ticks")
    mask = np.ones_like(wd)
    mask[np.tril_indices_from(mask)] = False
    annot_wd = [[f"10^{int(np.log10(abs(val)))}" if val != 0 else "0" for val in row] for row in wd]
    ax = sns.heatmap(wd, mask=mask, annot=annot_wd, fmt='', annot_kws={'size': 20}, cbar=True, linewidths=.5, cmap="RdYlGn_r", vmax=90, vmin=0)
    
    ax.set_yticklabels(Appname, rotation=0) 
    ax.set_xticklabels(Appname, ha="right", rotation=40) 
    ax.set_title("Wasserstein distance " + f)
    # ax.invert_yaxis()
    plt.savefig(f"{OUTPUT_DIR}/Wasserstein_distance_{f}.png")
    plt.close()

def simi_pca_mahalanobis(X, f):
    ''' 
    X is the data matrix, rows are applications, columns are events
    f is the path for X, used for graph title 
    ''' 
    Appname = [app_name_mapping.get(name, name) for name in X.index]
    X_value = X.values

    md = [[0] * len(X) for _ in range(len(X_value))]
    X_reduced = PCA(n_components=min(0.95, len(X_value) - 1)).fit_transform(X_value)
    t1 = time()
    mmd = distance.pdist(X_reduced, 'mahalanobis')
    
    k = 0
    for i in range(len(X_value)):
        for j in range(i + 1, len(X_value)):
            md[i][j] = md[j][i] = mmd[k] * 10
            k += 1
    t2 = time()
    print("cost time:", t2 - t1)
    plt.figure(figsize=(24, 18))
    sns.set(font_scale=2.5)
    sns.set_style("ticks")
    mask = np.ones_like(md)
    mask[np.tril_indices_from(mask)] = False
    ax = sns.heatmap(md, mask=mask, annot=True, fmt='.0f', annot_kws={'size': 20}, cbar=True, linewidths=.5, cmap="RdYlGn_r", vmax=90, vmin=0)
    
    ax.set_yticklabels(Appname, rotation=0) 
    ax.set_xticklabels(Appname, ha="right", rotation=40) 
    ax.set_title("pca + Mahalanobis " + f)
    # ax.invert_yaxis()
    plt.savefig(f"{OUTPUT_DIR}/pca_Mahalanobis_{f}.png")
    plt.close()

In [51]:
import csv

for app in test_results:
    groups = {}
    for trace in test_results[app]:
        pcap_df = test_results[app][trace]['PCAP']
        # Assume PCAP has a single row with a 'value' column
        pcap_value = pcap_df['value'].iloc[0]
        if pcap_value not in groups:
            groups[pcap_value] = []
        groups[pcap_value].append(trace) 
    
    averages_dict = {}
    for pcap_value, traces in groups.items():
        averages = {}
        for trace in traces:
            # Process derived_papi separately for each key
            if 'derived_papi' in test_results[app][trace]:
                derived_dict = test_results[app][trace]['derived_papi']
                for derived_key in ['TOT_INS_PER_CYC', 'TOT_CYC_PER_INS', 'L3_TCM_PER_TCA', 'TOT_STL_PER_CYC']:
                    if derived_key in derived_dict:
                        avg = derived_dict[derived_key]['value'].mean()
                        if derived_key not in averages:
                            averages[derived_key] = []
                        averages[derived_key].append(avg)
            
            # Process progress_per_cycle
            if 'progress_per_cycle' in test_results[app][trace]:
                df = test_results[app][trace]['progress_per_cycle']['progress_per_cycle']
                # Filter out inf and -inf values before computing mean
                valid_values = df['value'][~np.isinf(df['value'])]
                avg = valid_values.mean()
                if 'progress_per_cycle' not in averages:
                    averages['progress_per_cycle'] = []
                averages['progress_per_cycle'].append(avg)
        
        # Average the averages
        final_averages = {attr: np.mean(vals) for attr, vals in averages.items()}
        averages_dict[pcap_value] = final_averages
        # Save to CSV
        # with open(f'{OUTPUT_DIR}/{app}_{pcap_value}_averages.csv', 'w', newline='') as csvfile:
        #     writer = csv.writer(csvfile)
        #     writer.writerow(['attribute', 'average'])
        #     for attr, avg in final_averages.items():
        #         writer.writerow([attr, avg])

In [52]:
# import csv

# for app in test_results:
#     groups = {}
#     for trace in test_results[app]:
#         pcap_df = test_results[app][trace]['PCAP']
#         # Assume PCAP has a single row with a 'value' column
#         pcap_value = pcap_df['value'].iloc[0]
#         if pcap_value not in groups:
#             groups[pcap_value] = []
#         groups[pcap_value].append(trace) 
    
#     averages_dict = {}
#     for pcap_value, traces in groups.items():
#         averages = {}
#         for trace in traces:
#             for attr in ['derived_papi', 'progress_per_cycle']:
#                 if attr == 'energy':
#                     df = test_results[app][trace][attr]
#                     avg = df['value'].mean()
#                 elif attr == 'power':
#                     df = test_results[app][trace][attr]['average_power']
#                     avg = df['average_power'].mean()
#                 elif attr == 'progress':
#                     df = test_results[app][trace][attr]['progress_frequency_median']
#                     avg = df['median'].mean()
#                 elif attr == 'papi':
#                     papi_dict = test_results[app][trace][attr]
#                     avg = np.mean([df['instantaneous_value'].mean() for df in papi_dict.values()])
#                 elif attr == 'PCAP':
#                     df = test_results[app][trace][attr]
#                     avg = df['value'].mean()
#                 elif attr == 'derived_papi':
#                     derived_dict = test_results[app][trace][attr]
#                     avg = np.mean([df['value'].mean() for df in derived_dict.values()])
#                 elif attr == 'pubPower':
#                     df = test_results[app][trace][attr]
#                     avg = df['value'].mean()
#                 elif attr == 'progress_per_cycle':
#                     df = test_results[app][trace][attr]['progress_per_cycle']
#                     avg = df['value'].mean()
#                 if attr not in averages:
#                     averages[attr] = []
#                 averages[attr].append(avg)
#         # Average the averages
#         final_averages = {attr: np.mean(vals) for attr, vals in averages.items()}
#         averages_dict[pcap_value] = final_averages
#         # Save to CSV
#         with open(f'{OUTPUT_DIR}/{app}_{pcap_value}_averages.csv', 'w', newline='') as csvfile:
#             writer = csv.writer(csvfile)
#             writer.writerow(['attribute', 'average'])
#             for attr, avg in final_averages.items():
#                 writer.writerow([attr, avg])
    


In [53]:
for app in test_results:
    if averages_dict:
        sorted_averages_dict = dict(sorted(averages_dict.items(), key=lambda x: x[0]))
        X = pd.DataFrame.from_dict(sorted_averages_dict, orient='index')
        X = X.drop(columns=['energy', 'power', 'PCAP', 'papi'], errors='ignore')
        simi_cosine(X, app)
        simi_JS(X, app)
        simi_wd(X, app)
        simi_pca_mahalanobis(X, app)

cost time: 0.00029206275939941406
cost time: 0.0014791488647460938
cost time: 0.001499176025390625
cost time: 8.416175842285156e-05
cost time: 0.0001780986785888672
cost time: 0.0013790130615234375
cost time: 0.0015807151794433594
cost time: 7.510185241699219e-05
cost time: 0.00018095970153808594
cost time: 0.0014619827270507812
cost time: 0.0013890266418457031
cost time: 8.20159912109375e-05
cost time: 0.0002617835998535156
cost time: 0.0014448165893554688
cost time: 0.0014269351959228516
cost time: 7.891654968261719e-05
cost time: 0.00018405914306640625
cost time: 0.0014510154724121094
cost time: 0.0014011859893798828
cost time: 7.390975952148438e-05
cost time: 0.0001761913299560547
cost time: 0.0014650821685791016
cost time: 0.0014948844909667969
cost time: 7.605552673339844e-05
cost time: 0.00017499923706054688
cost time: 0.0014462471008300781
cost time: 0.0014891624450683594
cost time: 7.295608520507812e-05
cost time: 0.00021505355834960938
cost time: 0.0014641284942626953
cost ti

In [54]:
import numpy as np

# Define the two vectors from X.iloc[0] and X.iloc[1]
vec1 = np.array([4.254711e+07, 1.008993e+02, 1.870586e+02, 9.605244e+10, 1.010000e+02, 2.016683e+00, 1.015596e+02])
vec2 = np.array([5.192675e+07, 1.408591e+02, 2.005884e+02, 1.170107e+11, 1.410000e+02, 2.232890e+00, 1.409744e+02])

# Compute cosine similarity
cos_sim = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

print(f"Cosine similarity between the two vectors: {cos_sim}")

Cosine similarity between the two vectors: 0.9999999999996632


In [62]:
# Compute similarities across applications for each PCAP value
pcap_apps = {}
for app in test_results:
    for trace in test_results[app]:
        pcap_df = test_results[app][trace]['PCAP']
        pcap_value = pcap_df['value'].iloc[0]
        if pcap_value not in pcap_apps:
            pcap_apps[pcap_value] = {}
        if app not in pcap_apps[pcap_value]:
            pcap_apps[pcap_value][app] = []
        pcap_apps[pcap_value][app].append(trace)

for pcap_value, apps_dict in pcap_apps.items():
    averages_dict = {}
    for app, traces in apps_dict.items():
        averages = {}
        for trace in traces:
            # Process derived_papi separately for each key
            if 'derived_papi' in test_results[app][trace]:
                derived_dict = test_results[app][trace]['derived_papi']
                for derived_key in ['TOT_INS_PER_CYC', 'TOT_CYC_PER_INS', 'L3_TCM_PER_TCA', 'TOT_STL_PER_CYC']:
                    if derived_key in derived_dict:
                        avg = derived_dict[derived_key]['value'].mean()
                        if derived_key not in averages:
                            averages[derived_key] = []
                        averages[derived_key].append(avg)
            
            # Process other attributes
            for attr in ['energy', 'power', 'progress', 'papi', 'PCAP', 'pubPower']:
                if attr == 'energy':
                    df = test_results[app][trace][attr]
                    avg = df['value'].mean()
                elif attr == 'power':
                    df = test_results[app][trace][attr]['average_power']
                    avg = df['average_power'].mean()
                elif attr == 'progress':
                    df = test_results[app][trace][attr]['progress_frequency_median']
                    avg = df['median'].mean()
                elif attr == 'papi':
                    papi_dict = test_results[app][trace][attr]
                    avg = np.mean([df['instantaneous_value'].mean() for df in papi_dict.values()])
                elif attr == 'PCAP':
                    df = test_results[app][trace][attr]
                    avg = df['value'].mean()
                elif attr == 'pubPower':
                    df = test_results[app][trace][attr]
                    avg = df['value'].mean()
                if attr not in averages:
                    averages[attr] = []
                averages[attr].append(avg)
        
        final_averages = {attr: np.mean(vals) for attr, vals in averages.items()}
        averages_dict[app] = final_averages
    
    if len(averages_dict) > 1:
        X = pd.DataFrame.from_dict(averages_dict, orient='index')
        X = X.drop(columns=['energy', 'power', 'PCAP', 'papi'], errors='ignore')
        sorted_index = sorted(X.index, key=lambda x: (0 if 'stream' in x else 1 if 'npb' in x else 2, 1 if 'stream' in x and 'full' in x and 'phases' not in x else 0, x))
        X = X.reindex(sorted_index)
        simi_cosine(X, f'PCAP_{pcap_value}')
        simi_JS(X, f'PCAP_{pcap_value}')
        simi_wd(X, f'PCAP_{pcap_value}')
        simi_pca_mahalanobis(X, f'PCAP_{pcap_value}')

        X_value = X.values
        pca = PCA(n_components=min(0.95, len(X_value) - 1))
        X_reduced = pca.fit_transform(X_value)
        loadings = pca.components_[0]  # Loadings for PC1
        feature_names = X.columns
        top_feature = feature_names[abs(loadings).argmax()]
        print(f"For {app}, the attribute with maximum contribution to PC1 is: {top_feature}")

cost time: 0.0005042552947998047
cost time: 0.001271963119506836
cost time: 0.00125885009765625
cost time: 0.00010013580322265625
For ones-npb-mg, the attribute with maximum contribution to PC1 is: progress
cost time: 0.00017404556274414062
cost time: 0.0011749267578125
cost time: 0.0012509822845458984
cost time: 7.390975952148438e-05
For ones-npb-mg, the attribute with maximum contribution to PC1 is: progress
cost time: 0.0001919269561767578
cost time: 0.0011713504791259766
cost time: 0.0012552738189697266
cost time: 7.224082946777344e-05
For ones-npb-mg, the attribute with maximum contribution to PC1 is: progress
cost time: 0.0001971721649169922
cost time: 0.0011801719665527344
cost time: 0.0012888908386230469
cost time: 7.295608520507812e-05
For ones-npb-mg, the attribute with maximum contribution to PC1 is: progress
cost time: 0.0001850128173828125
cost time: 0.001148223876953125
cost time: 0.0012569427490234375
cost time: 7.295608520507812e-05
For ones-npb-mg, the attribute with m

In [63]:
# # Debug for PCAP 141.0
# if 141.0 in pcap_apps and len(pcap_apps[141.0]) > 1:
#     # Recalculate averages_dict for 141.0
#     averages_dict_debug = {}
#     for app, traces in pcap_apps[141.0].items():
#         averages = {}
#         for trace in traces:
#             for attr in ['energy', 'power', 'progress', 'papi', 'PCAP', 'derived_papi', 'pubPower']:
#                 if attr == 'energy':
#                     df = test_results[app][trace][attr]
#                     avg = df['value'].mean()
#                 elif attr == 'power':
#                     df = test_results[app][trace][attr]['average_power']
#                     avg = df['average_power'].mean()
#                 elif attr == 'progress':
#                     df = test_results[app][trace][attr]['progress_frequency_median']
#                     avg = df['median'].mean()
#                 elif attr == 'papi':
#                     papi_dict = test_results[app][trace][attr]
#                     avg = np.mean([df['instantaneous_value'].mean() for df in papi_dict.values()])
#                 elif attr == 'PCAP':
#                     df = test_results[app][trace][attr]
#                     avg = df['value'].mean()
#                 elif attr == 'derived_papi':
#                     derived_dict = test_results[app][trace][attr]
#                     avg = np.mean([df['value'].mean() for df in derived_dict.values()])
#                 elif attr == 'pubPower':
#                     df = test_results[app][trace][attr]
#                     avg = df['value'].mean()
#                 if attr not in averages:
#                     averages[attr] = []
#                 averages[attr].append(avg)
#         final_averages = {attr: np.mean(vals) for attr, vals in averages.items()}
#         averages_dict_debug[app] = final_averages
    
#     X_debug = pd.DataFrame.from_dict(averages_dict_debug, orient='index')
#     print("X for PCAP 141.0:")
#     print(X_debug)
#     cosX = cosine_similarity(X_debug)
#     print("Cosine similarity matrix:")
#     print(cosX)
    
#     # Optional: Simple clustering or ranking
#     from sklearn.cluster import KMeans
#     if len(X_debug) > 1:
#         kmeans = KMeans(n_clusters=min(3, len(X_debug)), random_state=42)
#         clusters = kmeans.fit_predict(X_debug)
#         print("Clusters based on attributes:")
#         for i, app in enumerate(X_debug.index):
#             print(f"{app}: Cluster {clusters[i]}")
